# DWARF-R1 C5 paired experiment — preflight only

**This notebook has no training or inference cell.** Running all cells only mounts Drive for reading, validates existing artifacts/data, and reports GPU availability. It never starts training, writes model files, or dispatches Factory work.

## Intended comparison
- **A:** verified C4 adapter + original C4 training data/recipe.
- **B:** the same C4 adapter + the same core data and budget, with the approved Factory Manager additions.
- Paired seeds requested: `182032`, `182033`, `182034`. Use exactly the same evaluation set and decode/scoring code for A and B.
- Metrics: accuracy, macro-F1, false acceptance, Work ID/Receipt validation, queue-decision accuracy, plus per-seed and aggregate mean/variance. Target accuracy is ≥85%, but any unauthorized false acceptance or failed deterministic validation blocks acceptance.

## Known blockers from existing evidence
- C4’s recorded split counts are 512/128/128, but its own manifest says synthetic policy families are shared across splits. Different IDs/layouts alone do not prove family-level isolation. The new benchmark needs a fresh, frozen, group-disjoint holdout.
- A prior saved C5 notebook contains an executed single-seed candidate run using seed `182033`; C4’s original run used `182032`. Those two seeds are already exposed and cannot honestly be described as untouched confirmatory seeds. The requested values are retained below as the requested plan, not asserted to be new.
- No separately identified, approved Factory Manager additions file has been pinned in this preflight. The prior C5 notebook generated synthetic examples from C4 prototypes; that is not a clean substitute for newly approved, group-disjoint data.
- GPU status must be freshly checked in a connected Colab runtime. Historical T4 output is not current readiness evidence.

**Do not train from this notebook.** Resolve the seed and dataset blockers, obtain fresh approval after preflight, and prepare a separate guarded training notebook only after all gates pass. Synthetic evaluation is not Android or Production acceptance.


In [ ]:
# Protocol constants only. No training is implemented in this notebook.
from pathlib import Path
import hashlib, json, zipfile, subprocess, sys

C4_ARCHIVE_NAME = "DWARF_R1_candidate4_model_and_evidence.zip"
C4_ARCHIVE_SIZE = 71_172_139
C4_ARCHIVE_SHA256 = "a23d3ae12e235f4cfe559848525179745bd040a2249661dd2290b3fe0cfdb4ff"
C4_ADAPTER_SHA256 = "f36716a0e1d9154abfa1aee5f61b418394745fccc8bb8e3c561d2941fe43e3aa"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
REQUESTED_SEEDS = (182032, 182033, 182034)
HISTORICALLY_EXPOSED_SEEDS = {182032, 182033}
EXPECTED_C4_COUNTS = {"train": 512, "validation": 128, "test": 128}
EXPECTED_C4_DATA_SHA256 = {
    "train": "0c29ea219992e038060f7d6eea6c26dfc7dbbb44937b3aa122267ce2d5fd2dba",
    "validation": "3041aad4847f0acd8969bad5996774b1e57b1afbc9118a570ee63ee34a2fa055",
    "test": "0dd180696efa851f2939ad46fc28e7a292f980d89f6d02eca759fee16f6ac538",
}

# C4 recipe recorded in the Factory Manager notebook: weights-only continuation, fresh optimizer.
REFERENCE_RECIPE = {
    "learning_rate": 1e-5, "optimizer": "AdamW", "weight_decay": 0.01,
    "train_rows": 512, "optimizer_steps": 128, "gradient_accumulation": 4,
    "per_example_microbatch": 1, "max_grad_norm": 1.0, "precision": "fp16",
    "grad_scaler_init_scale": 256, "gradient_checkpointing": True,
    "base_model": BASE_MODEL, "base_revision": BASE_REVISION,
}
TRAINING_ENABLED = False
HUMAN_APPROVAL_RECORDED = False
FACTORY_ADDITIONS_PATH = None  # Set only to a separately reviewed, approved dataset file.
print("PREPARATION ONLY — training enabled:", TRAINING_ENABLED)
print("Requested seeds:", REQUESTED_SEEDS, "| historically exposed:", sorted(HISTORICALLY_EXPOSED_SEEDS))
print("Reference recipe:", json.dumps(REFERENCE_RECIPE, indent=2))


In [ ]:
# Read existing Drive artifacts. This cell does not write to Drive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
C4_ARCHIVE = DRIVE_ROOT / C4_ARCHIVE_NAME
if not C4_ARCHIVE.is_file():
    candidates = list(DRIVE_ROOT.glob(C4_ARCHIVE_NAME))
    if len(candidates) == 1:
        C4_ARCHIVE = candidates[0]
    else:
        raise FileNotFoundError(f"Could not uniquely locate {C4_ARCHIVE_NAME} at MyDrive root; found {len(candidates)}")
print("C4 archive located:", C4_ARCHIVE)


In [ ]:
# Verify current archive bytes, ZIP CRC, internal file hashes, adapter, and dataset files.
def sha256_stream(stream):
    h = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        h.update(chunk)
    return h.hexdigest()

archive_sha = sha256_stream(C4_ARCHIVE.open("rb"))
archive_bytes = C4_ARCHIVE.stat().st_size
assert archive_bytes == C4_ARCHIVE_SIZE, (archive_bytes, C4_ARCHIVE_SIZE)
assert archive_sha == C4_ARCHIVE_SHA256, (archive_sha, C4_ARCHIVE_SHA256)
with zipfile.ZipFile(C4_ARCHIVE) as z:
    assert z.testzip() is None, "ZIP CRC failure"
    members = set(z.namelist())
    assert "SHA256SUMS.json" in members and "manifest.json" in members
    sums = json.loads(z.read("SHA256SUMS.json"))
    internal_bad = []
    for name, expected in sums.items():
        if name not in members:
            internal_bad.append((name, "missing")); continue
        actual = sha256_stream(z.open(name))
        if actual != expected:
            internal_bad.append((name, actual, expected))
    assert not internal_bad, internal_bad
    assert len(sums) == 39 and len(members) == 40, (len(sums), len(members))
    adapter_name = "adapter/adapter_model.safetensors"
    assert adapter_name in members
    adapter_sha = sha256_stream(z.open(adapter_name))
    assert adapter_sha == C4_ADAPTER_SHA256, (adapter_sha, C4_ADAPTER_SHA256)
    c4_manifest = json.loads(z.read("manifest.json"))
    split_bytes = {s: z.read(f"{s}.jsonl") for s in EXPECTED_C4_COUNTS}

print("ARCHIVE_SHA256_PASS", archive_sha, "bytes", archive_bytes)
print("ZIP_CRC_PASS; internal files:", len(sums), "/", len(members))
print("C4_ADAPTER_SHA256_PASS", adapter_sha)
print("C4 manifest counts:", c4_manifest.get("counts"))


In [ ]:
# Inspect split hashes, IDs, Work IDs, evidence refs, exact prompts, and known family-leakage warning.
def parse_jsonl(blob):
    return [json.loads(line) for line in blob.decode("utf-8").splitlines() if line.strip()]

rows_by_split = {s: parse_jsonl(b) for s, b in split_bytes.items()}
for split, rows in rows_by_split.items():
    digest = hashlib.sha256(split_bytes[split]).hexdigest()
    print(split, "rows=", len(rows), "sha256=", digest, "expected=", EXPECTED_C4_DATA_SHA256[split])
    assert len(rows) == EXPECTED_C4_COUNTS[split]
    assert digest == EXPECTED_C4_DATA_SHA256[split]
assert c4_manifest.get("counts") == EXPECTED_C4_COUNTS

all_ids, all_work_ids, all_evidence, all_prompts = {}, {}, {}, {}
for split, rows in rows_by_split.items():
    for row in rows:
        for label, key, value in [
            ("row id", all_ids, row.get("id")),
            ("work id", all_work_ids, row.get("snapshot", {}).get("workId")),
        ]:
            if value is not None:
                key.setdefault(value, set()).add(split)
        for ref in row.get("snapshot", {}).get("evidenceRefs", []):
            all_evidence.setdefault(ref, set()).add(split)
        all_prompts.setdefault(row.get("input", ""), set()).add(split)

def cross_split(values):
    return [value for value, splits in values.items() if len(splits) > 1]
exact_leakage = {
    "row_ids": cross_split(all_ids), "work_ids": cross_split(all_work_ids),
    "evidence_refs": cross_split(all_evidence), "identical_inputs": cross_split(all_prompts),
}
print("Exact cross-split reuse:", json.dumps({k: len(v) for k, v in exact_leakage.items()}))
assert not any(exact_leakage.values()), exact_leakage

limitation = str(c4_manifest.get("limitation", ""))
family_isolation_verified = "shared synthetic policy families" not in limitation.lower()
print("C4 recorded limitation:", limitation)
print("FAMILY_LEVEL_SPLIT_ISOLATION:", "PASS" if family_isolation_verified else "BLOCKED — shared synthetic policy families")


In [ ]:
# Fresh GPU probe. Historical notebook output is not reused as current evidence.
nvidia = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("nvidia-smi return code:", nvidia.returncode)
print(nvidia.stdout.strip() or nvidia.stderr.strip())
try:
    import torch
    gpu_ready = bool(torch.cuda.is_available() and torch.cuda.device_count() > 0)
    print("torch:", torch.__version__, "cuda_available:", torch.cuda.is_available())
    if gpu_ready:
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
except Exception as exc:
    gpu_ready = False
    print("torch/GPU probe failed:", repr(exc))
print("GPU_PREFLIGHT:", "PASS" if gpu_ready and nvidia.returncode == 0 else "BLOCKED — no live GPU evidence")


In [ ]:
# Gate summary. This notebook cannot start training, even if every probe passes.
seed_freshness_pass = not (set(REQUESTED_SEEDS) & HISTORICALLY_EXPOSED_SEEDS)
additions_present = bool(FACTORY_ADDITIONS_PATH and Path(FACTORY_ADDITIONS_PATH).is_file())
# A new frozen, group-disjoint evaluation set is mandatory; the old shared-family manifest fails this gate.
new_holdout_ready = False
checkpoint_ready = (archive_sha == C4_ARCHIVE_SHA256 and adapter_sha == C4_ADAPTER_SHA256)
gates = {
    "checkpoint_and_sha256": checkpoint_ready,
    "dataset_exact_integrity": True,
    "dataset_family_disjointness": family_isolation_verified,
    "approved_factory_manager_additions": additions_present,
    "fresh_unseen_evaluation_set": new_holdout_ready,
    "all_requested_seeds_unexposed": seed_freshness_pass,
    "live_gpu_ready": bool(gpu_ready),
    "human_training_approval_recorded": HUMAN_APPROVAL_RECORDED,
}
print(json.dumps(gates, indent=2))
print("PREFLIGHT:", "PASS" if all(gates.values()) else "BLOCKED")
print("TRAINING_ENABLED:", TRAINING_ENABLED, "— no training code exists in this notebook")
